# Set Up Enivironment


In [1]:
# [Cell 1: Check GPU]
!nvidia-smi

Sun Mar  8 17:02:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:4A:00.0 Off |                    0 |
| N/A   30C    P0             49W /  350W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# [Cell 2: Cleanup]
!rm -rf ./sample_data

# Preprocess Data

In [3]:
# [Cell 4: Directory Constants]
IMAGE_DIR = "./images"
DATASET_DIR = "./dataset"
VAL_DIR = "./val"
DATASET_DIR = "./dataset"

<B> SKIP THE CELL BELOW if you downloaded images via Kaggle CLI.


In [ ]:
# [Cell 5: Download & Extract Images]
# This cell is only needed if downloading directly from NIH Box.com URLs.
import os

IMAGE_DIR = "./images"
TRAIN_DIR = "./train"
VAL_DIR = "./val"
DATASET_DIR = "./dataset"

# Check if images already exist (either in ./images/ or already split into ./train/)
images_in_source = len(os.listdir(IMAGE_DIR)) if os.path.exists(IMAGE_DIR) else 0
images_in_train = len(os.listdir(TRAIN_DIR)) if os.path.exists(TRAIN_DIR) else 0

if images_in_source > 1000 or images_in_train > 1000:
    print(f"Images already exist!")
    print(f"  ./images/: {images_in_source} files")
    print(f"  ./train/:  {images_in_train} files")
    print("Skipping download. Delete these folders to re-download.")
else:
    import tarfile
    import urllib.request
    # URLs for the tar.gz files (6 of 12 archives from NIH)
    links = [
        'https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz',
        'https://nihcc.box.com/shared/static/i28rlmbvmfjbl8p2n3ril0pptcmcu9d1.gz',
        'https://nihcc.box.com/shared/static/f1t00wrtdk94satdfb9olcolqx20z2jp.gz',
        'https://nihcc.box.com/shared/static/0aowwzs5lhjrceb3qp67ahp0rd1l1etg.gz',
        'https://nihcc.box.com/shared/static/v5e3goj22zr6h8tzualxfsqlqaygfbsn.gz',
        'https://nihcc.box.com/shared/static/asi7ikud9jwnkrnkj99jnpfkjdes7l6l.gz',
    ]

    os.makedirs(DATASET_DIR, exist_ok=True)

    for index, link in enumerate(links):
        tar = "images_%02d.tar.gz" % (index + 1)
        tar_path = os.path.join(DATASET_DIR, tar)
        urllib.request.urlretrieve(link, tar_path)
        print(f"Downloaded {tar}")

        with tarfile.open(tar_path, "r") as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.lower().endswith(".png"):
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=IMAGE_DIR)
        print(f"Extracted {tar}")

        os.remove(tar_path)

    print("Download and extraction complete")

In [4]:
# [Cell 6: Imports & Constants] *** ALWAYS RUN ON RESTART ***
import os
import pandas as pd

IMAGE_DIR = "./images"
TRAIN_DIR = "./train"
VAL_DIR = "./val"
DATASET_DIR = "./dataset"
ORIGINAL_CSV = "./dataset/Data_Entry_2017.csv"
BINARY_MATRIX_CSV = "all_labels.csv"
TRAIN_CSV = "train_labels.csv"
VAL_CSV = "val_labels.csv"

print("Constants defined: IMAGE_DIR, TRAIN_DIR, VAL_DIR, DATASET_DIR, ORIGINAL_CSV, BINARY_MATRIX_CSV, TRAIN_CSV, VAL_CSV")

Constants defined: IMAGE_DIR, TRAIN_DIR, VAL_DIR, DATASET_DIR, ORIGINAL_CSV, BINARY_MATRIX_CSV, TRAIN_CSV, VAL_CSV


<B>SKIP CELL BELOW if all_labels.csv already exists

In [ ]:
# [Cell 7: Create Binary Label Matrix] *** SKIP if all_labels.csv already exists ***
df = pd.read_csv(ORIGINAL_CSV, usecols=["Image Index", "Finding Labels"])
df["Finding Labels"] = df["Finding Labels"].fillna("")

unique_labels_set = set()
labels_series = df["Finding Labels"].str.split("|")
for labels in labels_series:
    for label in labels:
        if label and label != "No Finding":
            unique_labels_set.add(label)
unique_labels = sorted(unique_labels_set)

for label in unique_labels:
    df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x.split("|") else 0)
binary_df = df[["Image Index"] + unique_labels]

binary_df.to_csv(BINARY_MATRIX_CSV, index=False)
print(f"Created {BINARY_MATRIX_CSV}")

<B> SKIP cell below if train_labels.csv and val_labels.csv already exist 

In [ ]:
# [Cell 8: Train/Val Split] *** SKIP if train_labels.csv and val_labels.csv already exist ***
# Creates train_df and val_df (needed by Cell 9 for symlinks)
RANDOM_SEED = 42

df = pd.read_csv(BINARY_MATRIX_CSV)
train_df = df.sample(frac=0.8, random_state=RANDOM_SEED)
val_df = df.drop(train_df.index)

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
print(f"Created {TRAIN_CSV} ({len(train_df)} rows) and {VAL_CSV} ({len(val_df)} rows)")

<b>SKIP cell below if ./train and ./val already have symlinks

In [5]:
# [Cell 9: Create Symlinks] *** SKIP if ./train and ./val already have symlinks (~20 min) ***
# Loads CSVs directly - no dependency on Cell 8's in-memory variables

train_images = pd.read_csv(TRAIN_CSV)["Image Index"]
val_images = pd.read_csv(VAL_CSV)["Image Index"]

if os.path.exists(TRAIN_DIR):
    for f in os.listdir(TRAIN_DIR):
        os.remove(os.path.join(TRAIN_DIR, f))
if os.path.exists(VAL_DIR):
    for f in os.listdir(VAL_DIR):
        os.remove(os.path.join(VAL_DIR, f))

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

abs_image_dir = os.path.abspath(IMAGE_DIR)

for image in train_images:
    src = os.path.join(abs_image_dir, image)
    dst = os.path.join(TRAIN_DIR, image)
    if os.path.exists(src):
        os.symlink(src, dst)

for image in val_images:
    src = os.path.join(abs_image_dir, image)
    dst = os.path.join(VAL_DIR, image)
    if os.path.exists(src):
        os.symlink(src, dst)

print(f"Created {len(os.listdir(TRAIN_DIR))} symlinks in {TRAIN_DIR}")
print(f"Created {len(os.listdir(VAL_DIR))} symlinks in {VAL_DIR}")

Created 89696 symlinks in ./train
Created 22424 symlinks in ./val


# Train

In [5]:
# [Cell 10: PyTorch Imports]
import torch
import torch.nn as nn
from torchvision import transforms, models, datasets

In [6]:
# [Cell 11: Training Hyperparameters]
num_epochs = 12
batch_size = 128
learning_rate = 2e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# [Cell 12: Dataset Class]
import pandas as pd
import PIL

class ChestXrayDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, csv_file, transform=None):
        self.images_dir = images_dir
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

        # Original names and labels
        image_names = self.annotations.iloc[:, 0].values
        labels = self.annotations.iloc[:, 1:].values.astype(float)

        # Keep only rows whose image file exists
        existing_mask = [
            os.path.exists(os.path.join(images_dir, img_name)) for img_name in image_names
        ]
        if not any(existing_mask):
            raise FileNotFoundError(
                f"No images found in {images_dir} that match entries in {csv_file}"
            )

        self.image_names = [name for name, keep in zip(image_names, existing_mask) if keep]
        self.labels = labels[existing_mask]

        self.num_classes = self.labels.shape[1]
        self.class_labels = list(self.annotations.columns)[1:]

        missing_count = len(image_names) - len(self.image_names)
        if missing_count > 0:
            print(
                f"ChestXrayDataset: Skipped {missing_count} missing files in '{images_dir}'. "
                f"Using {len(self.image_names)} images."
            )

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = os.path.join(self.images_dir, self.image_names[idx])
        # Double-check existence in case files changed after init
        if not os.path.exists(img_name):
            raise FileNotFoundError(f"Image no longer exists: {img_name}")
        image = PIL.Image.open(img_name).convert('RGB')
        labels = torch.FloatTensor(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, labels

In [9]:
# [Cell 13: Data Transforms]
# Resize, normalize to ImageNet stats for transfer learning
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [10]:
# [Cell 14: Create DataLoaders]
# Create datasets and dataloaders for training and validation
train_dataset = ChestXrayDataset(
    images_dir=TRAIN_DIR,
    csv_file=TRAIN_CSV,
    transform=transform
)
val_dataset = ChestXrayDataset(
    images_dir=VAL_DIR,
    csv_file=VAL_CSV,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

In [11]:
# [Cell 15: Model Definition]
# ResNet50 pretrained on ImageNet, with custom fully-connected layers for 14-class multilabel
num_labels = train_dataset.num_classes
class_labels = train_loader.dataset.class_labels
print(f"Number of labels: {num_labels}")
print(f"Labels: {class_labels}")

model = models.resnet50(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_labels)
)

# Unfreeze fc layers
for param in model.fc.parameters():
    param.requires_grad = True

# Use DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

model = model.to(device)

Number of labels: 14
Labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


/u50/moucattv/CXDD_model/cxdd_env/lib64/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/u50/moucattv/CXDD_model/cxdd_env/lib64/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using 4 GPUs with DataParallel!


In [12]:
# [Cell 16: Loss & Optimizer]
# BCEWithLogitsLoss for multilabel classification, Adam optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
# [Cell 17: Training Loop]
# Train with early stopping, checkpointing, progress tracking, and mixed precision
patience = 3
counter = 0
best_val_loss = float("inf")
best_model_path = "best_model.pth"

import time
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()
start_time = time.time()

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        # Progress update every 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation phase
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

    val_loss = val_running_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {epoch_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")

    # Checkpointing: save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save(model_to_save.state_dict(), best_model_path)
        print(f"  Saved new best model (val_loss: {val_loss:.4f})")
    else:
        counter += 1
        print(f"  No improvement ({counter}/{patience})")
        if counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    torch.cuda.empty_cache()

end_time = time.time()
print(f"\nTraining complete in {(end_time - start_time)/60:.1f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")

/tmp/ipykernel_2427053/2771136119.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_2427053/2771136119.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1 | Batch 100/701 | Loss: 0.2019
  Epoch 1 | Batch 200/701 | Loss: 0.1673
  Epoch 1 | Batch 300/701 | Loss: 0.1942
  Epoch 1 | Batch 400/701 | Loss: 0.1776
  Epoch 1 | Batch 500/701 | Loss: 0.2150
  Epoch 1 | Batch 600/701 | Loss: 0.1441
  Epoch 1 | Batch 700/701 | Loss: 0.2053


/tmp/ipykernel_2427053/2771136119.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/12]
  Train Loss: 0.1932
  Val Loss: 0.1738
  Saved new best model (val_loss: 0.1738)
  Epoch 2 | Batch 100/701 | Loss: 0.1725
  Epoch 2 | Batch 200/701 | Loss: 0.2057
  Epoch 2 | Batch 300/701 | Loss: 0.1783
  Epoch 2 | Batch 400/701 | Loss: 0.1645
  Epoch 2 | Batch 500/701 | Loss: 0.1515
  Epoch 2 | Batch 600/701 | Loss: 0.1707
  Epoch 2 | Batch 700/701 | Loss: 0.1647
Epoch [2/12]
  Train Loss: 0.1787
  Val Loss: 0.1713
  Saved new best model (val_loss: 0.1713)
  Epoch 3 | Batch 100/701 | Loss: 0.1960
  Epoch 3 | Batch 200/701 | Loss: 0.1820
  Epoch 3 | Batch 300/701 | Loss: 0.1852
  Epoch 3 | Batch 400/701 | Loss: 0.1824
  Epoch 3 | Batch 500/701 | Loss: 0.1653
  Epoch 3 | Batch 600/701 | Loss: 0.2039
  Epoch 3 | Batch 700/701 | Loss: 0.1567
Epoch [3/12]
  Train Loss: 0.1763
  Val Loss: 0.1701
  Saved new best model (val_loss: 0.1701)
  Epoch 4 | Batch 100/701 | Loss: 0.1872
  Epoch 4 | Batch 200/701 | Loss: 0.1673
  Epoch 4 | Batch 300/701 | Loss: 0.1958
  Epoch 4 | Batch 

In [14]:
# [Cell 18: Save Model]
# Best model is already saved as best_model.pth during training
# This cell saves a timestamped copy for version tracking
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
model_filename = f"{timestamp}_resnet50model.pth"

# Copy best model to timestamped file
import shutil
if os.path.exists("best_model.pth"):
    shutil.copy("best_model.pth", model_filename)
    print(f"Copied best model to: {model_filename}")
else:
    # Fallback: save current model state
    model_to_save = model.module if hasattr(model, 'module') else model
    torch.save(model_to_save.state_dict(), model_filename)
    print(f"Saved current model to: {model_filename}")

Copied best model to: 2026-03-07T17:48:49_resnet50model.pth


In [12]:
# [Cell 19: Load Model & Predict]
# Load a saved model and run inference on the dataset
def load_model(model_path, num_labels, device):
    model = models.resnet50(pretrained=False)
    # Must match the training architecture
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_labels)
    )
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model = model.to(device)
    return model

def predict(model, dataloader, device, class_labels, threshold=0.3):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs) > threshold
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Print the first 5 predictions with class labels
    for i in range(5):
        predicted_labels = [class_labels[j] for j in range(num_labels) if all_preds[i][j]]
        true_labels = [class_labels[j] for j in range(num_labels) if all_labels[i][j]]
        print(f"Image {i+1}:")
        print(f"  Predicted Labels: {predicted_labels}")
        print(f"  True Labels: {true_labels}")

# Load best model (or most recent timestamped model)
import glob

if os.path.exists("best_model.pth"):
    print("Loading best model: best_model.pth")
    model = load_model("best_model.pth", num_labels, device)
    predict(model, val_loader, device, class_labels)
else:
    model_files = glob.glob("*_resnet50model.pth")
    if model_files:
        latest_model = max(model_files, key=os.path.getctime)
        print(f"Loading model: {latest_model}")
        model = load_model(latest_model, num_labels, device)
        predict(model, val_loader, device, class_labels)
    else:
        print("No model files found. Please train the model first.")

Loading best model: best_model.pth


/u50/moucattv/CXDD_model/cxdd_env/lib64/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Image 1:
  Predicted Labels: []
  True Labels: ['Cardiomegaly', 'Effusion']
Image 2:
  Predicted Labels: []
  True Labels: ['Hernia']
Image 3:
  Predicted Labels: []
  True Labels: []
Image 4:
  Predicted Labels: []
  True Labels: []
Image 5:
  Predicted Labels: []
  True Labels: ['Effusion', 'Infiltration']


In [13]:
# [Cell 20: Comprehensive Model Evaluation]
# Compute detailed metrics on validation set with confusion matrix stats
import numpy as np
from sklearn.metrics import roc_auc_score

def evaluate_model(model, dataloader, device, class_labels, threshold=0.3):
    model.eval()
    all_probs = []
    all_preds = []
    all_labels = []
    
    print("Running inference on validation set...")
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = probs > threshold
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    n_samples = len(all_labels)
    
    # =====================================================================
    # PER-CLASS METRICS WITH CONFUSION MATRIX
    # =====================================================================
    print("\n" + "="*100)
    print("PER-CLASS METRICS")
    print("="*100)
    print(f"{'Disease':<20} {'TP':>6} {'FP':>6} {'TN':>6} {'FN':>6} | {'Prec':>6} {'Recall':>6} {'F1':>6} {'AUC':>6} | {'Support':>7}")
    print("-"*100)
    
    # Store for overall calculations
    total_tp, total_fp, total_tn, total_fn = 0, 0, 0, 0
    all_precision, all_recall, all_f1 = [], [], []
    auc_scores = []
    
    for i, label in enumerate(class_labels):
        y_true = all_labels[:, i]
        y_pred = all_preds[:, i]
        y_prob = all_probs[:, i]
        
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        support = tp + fn
        
        total_tp += tp
        total_fp += fp
        total_tn += tn
        total_fn += fn
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        if support > 0:
            all_precision.append(precision)
            all_recall.append(recall)
            all_f1.append(f1)
        
        try:
            auc = roc_auc_score(y_true, y_prob)
            auc_scores.append(auc)
        except:
            auc = 0.0
        
        print(f"{label:<20} {tp:>6} {fp:>6} {tn:>6} {fn:>6} | {precision:>6.3f} {recall:>6.3f} {f1:>6.3f} {auc:>6.3f} | {support:>7}")
    
    # =====================================================================
    # OVERALL CONFUSION MATRIX PERCENTAGES
    # =====================================================================
    print("\n" + "="*100)
    print("OVERALL CONFUSION MATRIX (aggregated across all classes)")
    print("="*100)
    total_predictions = total_tp + total_fp + total_tn + total_fn
    print(f"                    Predicted Positive    Predicted Negative")
    print(f"  Actual Positive      TP: {total_tp:>7} ({100*total_tp/total_predictions:>5.2f}%)     FN: {total_fn:>7} ({100*total_fn/total_predictions:>5.2f}%)")
    print(f"  Actual Negative      FP: {total_fp:>7} ({100*total_fp/total_predictions:>5.2f}%)     TN: {total_tn:>7} ({100*total_tn/total_predictions:>5.2f}%)")
    
    # =====================================================================
    # OVERALL PRECISION, RECALL, F1
    # =====================================================================
    print("\n" + "="*100)
    print("OVERALL METRICS")
    print("="*100)
    
    # Micro-averaged (pool all TP/FP/FN)
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0
    
    print(f"\nMICRO-AVERAGED (treats all predictions equally):")
    print(f"  Precision: {micro_precision:.4f}")
    print(f"  Recall:    {micro_recall:.4f}")
    print(f"  F1 Score:  {micro_f1:.4f}")
    
    # Macro-averaged (average of per-class metrics)
    macro_precision = np.mean(all_precision) if all_precision else 0
    macro_recall = np.mean(all_recall) if all_recall else 0
    macro_f1 = np.mean(all_f1) if all_f1 else 0
    macro_auc = np.mean(auc_scores) if auc_scores else 0
    
    print(f"\nMACRO-AVERAGED (average across diseases, treats each disease equally):")
    print(f"  Precision: {macro_precision:.4f}")
    print(f"  Recall:    {macro_recall:.4f}")
    print(f"  F1 Score:  {macro_f1:.4f}")
    print(f"  AUC-ROC:   {macro_auc:.4f}")
    
    # =====================================================================
    # SAMPLE PREDICTIONS
    # =====================================================================
    print("\n" + "="*100)
    print("SAMPLE PREDICTIONS (first 5)")
    print("="*100)
    for i in range(min(5, len(all_preds))):
        pred_labels = [class_labels[j] for j in range(len(class_labels)) if all_preds[i][j]]
        true_labels_list = [class_labels[j] for j in range(len(class_labels)) if all_labels[i][j]]
        print(f"Image {i+1}:")
        print(f"  Predicted: {pred_labels if pred_labels else ['(none)']}")
        print(f"  Actual:    {true_labels_list if true_labels_list else ['(none)']}")
    
    return all_probs, all_preds, all_labels

# Run evaluation
all_probs, all_preds, all_labels = evaluate_model(model, val_loader, device, class_labels)


Running inference on validation set...

PER-CLASS METRICS
Disease                  TP     FP     TN     FN |   Prec Recall     F1    AUC | Support
----------------------------------------------------------------------------------------------------
Atelectasis               5     12  20066   2341 |  0.294  0.002  0.004  0.728 |    2346
Cardiomegaly              0      0  21897    527 |  0.000  0.000  0.000  0.733 |     527
Consolidation             0      0  21470    954 |  0.000  0.000  0.000  0.755 |     954
Edema                     0      0  21945    479 |  0.000  0.000  0.000  0.843 |     479
Effusion                854   1536  18303   1731 |  0.357  0.330  0.343  0.792 |    2585
Emphysema                 0      0  21909    515 |  0.000  0.000  0.000  0.814 |     515
Fibrosis                  0      0  22121    303 |  0.000  0.000  0.000  0.715 |     303
Hernia                    0      0  22386     38 |  0.000  0.000  0.000  0.774 |      38
Infiltration            933   1581  1686

In [20]:
# [Diagnostic: Check model output probabilities]
import torch
import numpy as np

model.eval()
with torch.no_grad():
    # Get one batch
    images, labels = next(iter(val_loader))
    images = images.to(device)
    outputs = model(images)
    probs = torch.sigmoid(outputs)
    
    print("=" * 70)
    print("DIAGNOSTIC: Raw Model Outputs")
    print("=" * 70)
    
    # Statistics
    print(f"\nRaw outputs (before sigmoid):")
    print(f"  Min: {outputs.min().item():.4f}")
    print(f"  Max: {outputs.max().item():.4f}")
    print(f"  Mean: {outputs.mean().item():.4f}")
    
    print(f"\nProbabilities (after sigmoid):")
    print(f"  Min: {probs.min().item():.4f}")
    print(f"  Max: {probs.max().item():.4f}")
    print(f"  Mean: {probs.mean().item():.4f}")
    
    # Per-class max probabilities
    print(f"\nPer-class MAX probability across batch:")
    for i, label in enumerate(class_labels):
        max_prob = probs[:, i].max().item()
        print(f"  {label:<25}: {max_prob:.4f}")
    
    # Count predictions at different thresholds
    print(f"\nPredictions at different thresholds:")
    for thresh in [0.5, 0.3, 0.2, 0.1, 0.05]:
        count = (probs > thresh).sum().item()
        print(f"  Threshold {thresh}: {count} positive predictions")

DIAGNOSTIC: Raw Model Outputs

Raw outputs (before sigmoid):
  Min: -11.8131
  Max: 0.3039
  Mean: -3.3498

Probabilities (after sigmoid):
  Min: 0.0000
  Max: 0.5754
  Mean: 0.0702

Per-class MAX probability across batch:
  Atelectasis              : 0.2864
  Cardiomegaly             : 0.1101
  Consolidation            : 0.1702
  Edema                    : 0.1603
  Effusion                 : 0.5754
  Emphysema                : 0.2352
  Fibrosis                 : 0.0818
  Hernia                   : 0.0272
  Infiltration             : 0.4162
  Mass                     : 0.1508
  Nodule                   : 0.1764
  Pleural_Thickening       : 0.1172
  Pneumonia                : 0.0680
  Pneumothorax             : 0.3508

Predictions at different thresholds:
  Threshold 0.5: 4 positive predictions
  Threshold 0.3: 47 positive predictions
  Threshold 0.2: 145 positive predictions
  Threshold 0.1: 428 positive predictions
  Threshold 0.05: 799 positive predictions
